### Machine Learning Models Comparison using:
### Model Training 
#### 1. Logistic Regression
#### 2. SVM
#### 3. Random Forest 
#### 4. XGboost 

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')  
from src.evaluate import evaluate_model, evaluate_threshold, plot_feature_importance_xgb
from src.visualization.plotting import visualize_threshold_results

df = pd.read_csv('../data/cleaned_encoded_data.csv')
df.isna().sum()
df.info()

In [ ]:
# Split Features and Target
X = df.drop(columns=['no_show', 'icd'])
y = df['no_show']


In [ ]:
# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42) # Stratify to maintain the distribution of the target variable in both train and test sets
feature_names = X_train.columns.tolist()
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from sklearn.preprocessing import StandardScaler


In [ ]:
### Add Imputer and Missing indicators for missing values
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# Create imputer: missing value indicator + imputed features
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Feature scaling: Logistic Regression, SVM, Neural Network
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Without Feature Scaling: Random Forest, XGBoost
X_train_final = pd.DataFrame(X_train_imputed, columns=feature_names)
X_test_final = pd.DataFrame(X_test_imputed, columns=feature_names)

# Final Features: X_train_final, X_test_final (Imbalanced)


In [ ]:
X_train_final.shape, X_test_final.shape

In [ ]:
# Fix Imbalance data using SMOTE: X_train_balanced, y_train_balanced
print("Before SMOTE:")
print(y_train.value_counts())

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_final, y_train)

print("After SMOTE:")
print(y_train_balanced.value_counts())

#### 1. X_train_final: Original imbalanced data with imputation and missing indicators, pair with Random Forest using class_weight#
#### 2. X_train_balanced: Balanced data using SMOTE: includes synthetic data, might not reflect true conditions

In [ ]:
from sklearn.metrics import make_scorer
# 1. 定義 cost savings 計算函數
def calculate_cost_savings(y_true, y_pred):
    """
    計算成本節省
    - True Positive: 正確預測不會來的病人，節省 200（未使用的時段可以重新安排）
    - False Positive: 錯誤預測不會來的病人，損失 20（不必要的干預成本）
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    no_show_cost = 200       # 未能預測到的未出現成本
    intervention_cost = 20    # 干預成本
    
    savings = (tp * no_show_cost) - ((fp + tp) * intervention_cost)
    return savings

# 2. 創建自定義評分器
cost_savings_scorer = make_scorer(calculate_cost_savings, greater_is_better=True)
scoring = {
    'recall': 'recall',
    'cost_savings': cost_savings_scorer
}

def custom_refit_strategy(cv_results):
    # 自定義選擇最佳參數的邏輯
    cost_savings = cv_results['mean_test_cost_savings']
    recall = cv_results['mean_test_recall']
    # 綜合考慮兩個指標
    combined_score = 0.7 * cost_savings + 0.3 * recall
    return np.argmax(combined_score)

## Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_balanced, y_train_balanced)

In [ ]:
y_pred_lg = logreg.predict(X_test_final)
y_proba_lg = logreg.predict_proba(X_test_final)[:, 1]

#### Evaluation Metrics:
1. Accuracy = (TP + TN) / (TP + FP + TN + FN) 
2. Precision = TP / (TP + FN) 
3. Recall = TP / (TP + EN) -> 成功預測出來的比例
4. F1 Score = 2 * (Precision + Recall) / (Precision + Recall) 
5. ROC AUC

In [ ]:
evaluate_model(y_test, y_pred_lg, y_proba_lg, "Logistic Regression")

In [ ]:
evaluate_threshold(y_test, y_proba_lg)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from sklearn.metrics import recall_score, precision_score, roc_auc_score
# Fine-Tuning Logistic Regression
# 1. 定義參數分佈
lr_param_dist = {
    'C': uniform(0.001, 100),              # 連續分佈從0.001到100
    'class_weight': ['balanced', None],     # 類別權重
    'max_iter': [1000],                     # 固定最大迭代次數
    'penalty': ['l1', 'l2'],                # 正則化類型
    'solver': ['liblinear', 'saga']         # 適用於 l1 和 l2 的求解器
}

# 2. 設置 RandomizedSearchCV
random_search_lr = RandomizedSearchCV(
    LogisticRegression(),
    param_distributions=lr_param_dist,
    n_iter=20,                # 隨機搜索的次數
    cv=5,                     # 5-fold cross validation
    scoring=scoring,         # 使用 recall 作為評分標準
    n_jobs=-1,               # 使用所有可用的 CPU
    verbose=1,               # 顯示進度
    random_state=42,        # 設置隨機種子以確保可重複性
    refit=custom_refit_strategy
)

# 3. 擬合模型
random_search_lr.fit(X_train_scaled, y_train)

In [ ]:

print("Best Parameters:", random_search_lr.best_params_)

best_lr = random_search_lr.best_estimator_
y_pred_lg = best_lr.predict(X_test_scaled)
y_proba_lg = best_lr.predict_proba(X_test_scaled)[:, 1]

evaluate_model(y_test, y_pred_lg, y_proba_lg, "Logistic Regression_Randomized_Search")

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. 在最佳參數附近設置更精細的搜索範圍
lr_params_fine = {
    'C': [20,30,40],          
    'class_weight': ['balanced'],              
    'max_iter': [1000],                     
    'penalty': ['l1'],                         
    'solver': ['liblinear']                       
}

# 2. 設置 GridSearchCV
grid_search_lr_fine = GridSearchCV(
    LogisticRegression(),
    param_grid=lr_params_fine,
    cv=5,                     # 5-fold cross validation
    scoring=scoring,         # 使用 recall 作為評分標準
    n_jobs=-1,               # 使用所有可用的 CPU
    verbose=1,                # 顯示進度
    refit= custom_refit_strategy
)

# 3. 擬合模型
grid_search_lr_fine.fit(X_train_scaled, y_train)


In [ ]:
print("Best Parameters:", grid_search_lr_fine.best_params_)

# Predict using best model
best_lr = grid_search_lr_fine.best_estimator_
y_pred_lg_best = best_lr.predict(X_test_scaled)
y_proba_lg_best = best_lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate best model
evaluate_model(y_test, y_pred_lg_best, y_proba_lg_best, "Logistic Regression_Fine_Tuned")
results_df = evaluate_threshold(y_test, y_proba_lg_best)
visualize_threshold_results(results_df)

## SVM

In [ ]:
from sklearn.svm import SVC

# Define and train SVM model
svm_model = SVC(
    kernel='rbf',  # Use RBF kernel function
    probability=True,  # Need this to get probability predictions
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score
# Use GridSearchCV to find best parameters
param_grid_svm = {
    'C': [1],           # Use default value
    'gamma': ['scale'], # Use default value
    'class_weight': ['balanced', {0:1, 1:10}]  # Test two weight options
}

grid_svm = GridSearchCV(
    svm_model,
    param_grid_svm,
    scoring=scoring,
    refit=custom_refit_strategy,
    cv=3,  
    n_jobs=-1,
    verbose=2
)

# Train model
print("Training SVM model...")
grid_svm.fit(X_train_scaled, y_train)




In [ ]:
# Get best model
best_svm = grid_svm.best_estimator_

# Predict
y_pred_svm = best_svm.predict(X_test_scaled)
y_proba_svm = best_svm.predict_proba(X_test_scaled)[:, 1]

# Evaluate model
print("\nSVM Model Evaluation:")
print("Best parameters:", grid_svm.best_params_)

from src.evaluate import evaluate_model
evaluate_model(y_test, y_pred_svm, y_proba_svm, "SVM")
results_df = evaluate_threshold(y_test, y_proba_svm)
visualize_threshold_results(results_df)

## Random Forest 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Model 1: Random Forest + SMOTE
rf_model = RandomForestClassifier(
    n_estimators=100,        # number of trees
    max_depth=None,          # allow trees to grow fully
    random_state=42,
    class_weight=None        # SMOTE already handled imbalance
)

rf_model.fit(X_train_balanced, y_train_balanced)


In [ ]:
y_pred_rf = rf_model.predict(X_test_final)
y_proba_rf = rf_model.predict_proba(X_test_final)[:, 1]  # needed for AUC

In [ ]:
## Initial Model
evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")
# Determine the best threshold for cost saving
results_df = evaluate_threshold(y_test, y_proba_rf)
visualize_threshold_results(results_df)

In [ ]:
## Weighted models (Dealing with the Imbalanced Dataset)
rf_model_2 = RandomForestClassifier(
    n_estimators=200,        # number of trees
    max_depth=None,          # allow trees to grow fully
    random_state=42,
    class_weight='balanced',        
    min_samples_split=2
)

rf_model_2.fit(X_train_final, y_train)
y_pred_rf_weighted = rf_model_2.predict(X_test_final)
y_proba_rf_weighted = rf_model_2.predict_proba(X_test_final)[:, 1]  # needed for AUC

In [ ]:
evaluate_model(y_test, y_pred_rf_weighted, y_proba_rf_weighted, "Random Forest_Weighted")

# Determine the best threshold for cost saving
results_df = evaluate_threshold(y_test, y_proba_rf_weighted)
visualize_threshold_results(results_df)


### Weighted RF seems to better than using SMOTE


In [ ]:
# Hyperparameter Tuning
# 1. Grid Search Paramters
param_grid_rf_fine = {
    'n_estimators': [180, 200, 220],  
    'max_depth': [None, 30, 40],      # Test if depth is needed
    'min_samples_split': [2, 3],      # Fine-tune split conditions
    'min_samples_leaf': [1, 2],       # Control leaf node size
    'max_features': ['sqrt', 'log2'],  # Feature selection method
    'class_weight': ['balanced'],
}

# Define Cost Saving to used as Scoring 
def calculate_cost_savings(cm_values):
    """Calculate cost savings"""
    tn, fp, fn, tp = cm_values
    savings = tp * 200  # Each correct no-show prediction saves $200
    costs = (fp * 20)   # Each false alarm costs $20
    missed = fn * 200   # Each missed no-show costs $200
    return savings - costs - missed

# 2. Use multiple evaluation metrics for GridSearchCV
grid_rf_fine = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf_fine,
    scoring=scoring,
    refit=custom_refit_strategy,  # Use cost savings as selection standard
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Start fine-tuning...")
grid_rf_fine.fit(X_train_final, y_train)

In [ ]:

# Get best model
best_rf = grid_rf_fine.best_estimator_
print("\nBest parameters:", grid_rf_fine.best_params_)

# Evaluate best model
y_pred_rf_weighted_best = best_rf.predict(X_test)
y_proba_rf_weighted_best = best_rf.predict_proba(X_test)[:, 1]

# Detailed evaluation
print("\nBest model evaluation:")
evaluate_model(y_test, y_pred_rf_weighted_best, y_proba_rf_weighted_best, "RF_Fine_Tuned")

# Threshold optimization
results_df = evaluate_threshold(y_test, y_proba_rf_weighted_best)
print("\nThreshold optimization results:")
print(results_df.sort_values('cost_savings', ascending=False).head())

visualize_threshold_results(results_df)

# Feature Importance




In [ ]:
threshold = 0.20
y_pred_custom = (y_proba_rf_weighted_best >= threshold).astype(int)
evaluate_model(y_test, y_pred_custom, y_proba_rf_weighted_best, "RF_Fine_Tuned_Custom_Threshold_0.25")


## No differences between weighted and SMOTE
1. SMOTE creates synthetic examples that may not generalize well

2. Random Forest already does internal bootstrapping + averaging

3. Giving the model correct penalties via class weights is more elegant and stable



In [ ]:
grid_rf_fine.fit(X_train_final, y_train)

In [ ]:
# Get best model
best_rf = grid_rf_fine.best_estimator_
print("\nBest parameters:", grid_rf_fine.best_params_)

# Evaluate best model
y_pred_rf_weighted_best = best_rf.predict(X_test_final)
y_proba_rf_weighted_best = best_rf.predict_proba(X_test_final)[:, 1]

# Detailed evaluation
print("\nBest model evaluation:")
evaluate_model(y_test, y_pred_rf_weighted_best, y_proba_rf_weighted_best, "RF_Fine_Tuned")

# Threshold optimization   
results_df = evaluate_threshold(y_test, y_proba_rf_weighted_best)
print("\nThreshold optimization results:")
print(results_df.sort_values('cost_savings', ascending=False).head())

visualize_threshold_results(results_df)

In [ ]:
threshold = 0.20
y_pred_custom = (y_proba_rf_weighted_best >= threshold).astype(int)
evaluate_model(y_test, y_pred_custom, y_proba_rf_weighted_best, "RF_Fine_Tuned_Custom_Threshold_0.30")

In [ ]:
# Feature Importance
import matplotlib.pyplot as plt
import seaborn as sns
def plot_feature_importance_rf(rf_model, X_train, top_n=15):
    # 1. 獲取特徵重要性
    importances = rf_model.feature_importances_
    feature_names = X_train.columns
    
    # 2. 創建DataFrame並排序
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    feature_importance = feature_importance.sort_values('importance', ascending=False)
    
    # 3. 選擇前N個特徵
    if top_n:
        feature_importance = feature_importance.head(top_n)
    
    # 4. 繪圖
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=feature_importance,
        x='importance',
        y='feature',
        palette='viridis'
    )
    
    plt.title('Feature Importance (Random Forest)', fontsize=15, pad=15)
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    
    # 5. 添加數值標籤
    for i, v in enumerate(feature_importance['importance']):
        plt.text(v, i, f'{v:.3f}', va='center')
    
    plt.tight_layout()
    plt.show()

# 使用方法
plot_feature_importance_rf(best_rf, X_train)

### XGBoost

In [ ]:
import xgboost as xgb

In [ ]:
# Compute ratio: (negative class / positive class) for imbalanced data
# So that the model pays more attention to the minority class
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", ratio)


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=ratio,    
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_final, y_train)


In [ ]:
y_pred_xgb = xgb_model.predict(X_test_final)
y_proba_xgb = xgb_model.predict_proba(X_test_final)[:, 1]

evaluate_model(y_test, y_pred_xgb, y_proba_xgb, "XGBoost")

## For Shows (Class 0):
1. Precision: 0.97 → 97% of patients predicted to show up actually showed up (improved)
2. Recall: 0.44 → Only 44% of actual show-ups were correctly identified (decreased)
3. F1-score: 0.61 → Moderate balanced performance for show-ups

## For No-shows (Class 1):
1. Precision: 0.15 → Only 15% of patients predicted to no-show actually didn't show up
2. Recall: 0.86 → 86% of actual no-shows were correctly identified (significantly improved)
3. F1-score: 0.25 → Still weak overall performance, but better at catching no-shows

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'min_child_weight': [1, 3],    # Add to prevent overfitting
    'reg_alpha': [0.1, 1.0],       # L1 regularization
    'reg_lambda': [1.0, 5.0]       # L2 regularization
}

xgb_clf = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

# Use multiple evaluation metrics for GridSearchCV
grid_xgb_fine = GridSearchCV(
    xgb_clf,                   # Use the XGBoost classifier defined above
    param_grid,
    scoring=scoring,
    refit=custom_refit_strategy,  # Use cost savings as selection standard
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [ ]:
# grid_search.fit(X_train, y_train)
grid_xgb_fine.fit(X_train_final, y_train)

In [ ]:
best_xgb = grid_xgb_fine.best_estimator_

print("Best parameters:", grid_xgb_fine.best_params_)

# Predict and evaluate
y_pred_xgb_best = best_xgb.predict(X_test_final)
y_proba_xgb_best = best_xgb.predict_proba(X_test_final)[:, 1]

evaluate_model(y_test, y_pred_xgb_best, y_proba_xgb_best, "XGBoost_Best")
results_df = evaluate_threshold(y_test, y_proba_xgb_best)
visualize_threshold_results(results_df)


In [ ]:

threshold = 0.45
y_pred_thresholded = (y_proba_xgb_best >= threshold).astype(int)
evaluate_model(y_test, y_pred_thresholded, y_proba_xgb_best, "XGBoost_Best")


In [ ]:
# # Feature Importance

plot_feature_importance_xgb(best_xgb)

In [ ]:
# Top features based on feature importance score
# Select top 15 features
importances = best_xgb.get_booster().get_score(importance_type='weight')
top_features = sorted(importances, key=importances.get, reverse=True)[:15]
print(top_features)

# Create a dataset with only top features
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]


# Train model with just these features
xgb_top = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

xgb_top.fit(X_train_top, y_train)

# Compare performance with full-feature model
y_pred_top_xgb = xgb_top.predict(X_test_top)
print(classification_report(y_test, y_pred_top_xgb))
y_proba_top_xgb = xgb_top.predict_proba(X_test_top)[:, 1]

evaluate_model(y_test, y_pred_top_xgb, y_proba_top_xgb, "XGBoost_Top_Features")
results_df = evaluate_threshold(y_test, y_proba_top_xgb)
visualize_threshold_results(results_df)


In [ ]:
threshold = 0.45
y_pred_thresholded = (y_proba_top_xgb >= threshold).astype(int)

print("Threshold-Tuned Report:")
print(classification_report(y_test, y_pred_thresholded))


In [ ]:
# Further Enhance Model Performance by creating interactions between features
# Create interactions between top features
X_train_enhanced = X_train[top_features].copy()
X_test_enhanced = X_test[top_features].copy()

# Weather x Age interactions
X_train_enhanced['temp_age_interaction'] = X_train['max_temp_day'] * X_train['age']
X_test_enhanced['temp_age_interaction'] = X_test['max_temp_day'] * X_test['age']

X_train_enhanced['rain_age_interaction'] = X_train['average_rain_day'] * X_train['age']
X_test_enhanced['rain_age_interaction'] = X_test['average_rain_day'] * X_test['age']

# Month x Weather interactions
for month in ['appointment_month_6', 'appointment_month_7', 'appointment_month_8']:
    X_train_enhanced[f'{month}_temp'] = X_train[month] * X_train['max_temp_day']
    X_test_enhanced[f'{month}_temp'] = X_test[month] * X_test['max_temp_day']
    
    X_train_enhanced[f'{month}_rain'] = X_train[month] * X_train['average_rain_day']
    X_test_enhanced[f'{month}_rain'] = X_test[month] * X_test['average_rain_day']



In [ ]:
# Focus parameter grid on feature-related parameters
focused_param_grid = {
    'n_estimators': [200, 300],          # Reduced options
    'max_depth': [4, 6],                 # Reduced options
    'learning_rate': [0.05, 0.1],        # Smaller learning rates
    'colsample_bytree': [0.6, 0.7, 0.8], # Focus on feature sampling
    'subsample': [0.8, 0.9],             # Less row sampling variation
    'gamma': [0.1, 0.3, 0.5],            # More focus on feature split quality
    'min_child_weight': [1, 3]           # Fewer options
}

# Create model for grid search
xgb_focused = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

# Run grid search
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with custom metric focusing on recall
from sklearn.metrics import make_scorer, fbeta_score
custom_scorer = make_scorer(fbeta_score, beta=2)

grid_search_focused = GridSearchCV(
    estimator=xgb_focused,
    param_grid=focused_param_grid,
    scoring=custom_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit on enhanced features dataset
grid_search_focused.fit(X_train_enhanced, y_train)

In [ ]:
# Get best model from grid search
best_model = grid_search_focused.best_estimator_

print("Best parameters:", grid_search_focused.best_params_)

# Predict and evaluate
y_pred_best_enhanced = best_model.predict(X_test_enhanced)
y_proba_best_enhanced = best_model.predict_proba(X_test_enhanced)[:, 1]

evaluate_model(y_test, y_pred_best_enhanced, y_proba_best_enhanced, "XGBoost_Best")
results_df = evaluate_threshold(y_test, y_proba_best_enhanced)
visualize_threshold_results(results_df)



In [ ]:
threshold = 0.45
y_pred_thresholded = (y_proba_best_enhanced >= threshold).astype(int)

print("Threshold-Tuned Report:")
print(classification_report(y_test, y_pred_thresholded))



### DNN

In [ ]:
from src.training.nn_trainer import run_nn_pipeline
# 定義不同的配置進行實驗
configs = [
    {
        "learning_rate": 0.001,
        "weight_decay": 0.01,
        "batch_size": 64,
        "hidden_dims": [512, 256, 128, 64],
        "dropout_rates": [0.5, 0.4, 0.3, 0.2],
        "epochs": 10,
        "patience": 15,
        "pos_weight": 8395/932  # 添加這個參數
    },
    {
        "learning_rate": 0.0005,
        "weight_decay": 0.02,
        "batch_size": 128,
        "hidden_dims": [512, 256, 128],
        "dropout_rates": [0.5, 0.4, 0.3],
        "epochs": 30,
        "patience": 10,
        "pos_weight": 8395/932  # 添加這個參數
    }
]
numerical_features = [
        'age', 
        'max_temp_day', 
        'average_temp_day', 
        'average_rain_day',
        'max_rain_day'
    ]
results = {}
# 運行實驗
for i, config in enumerate(configs):
    print(f"\nTraining Model with Config {i+1}:")
    model, best_threshold, final_metrics, val_probs, val_labels = run_nn_pipeline(X, y, numerical_features, config)
    
    results[f"Config_{i+1}"] = {
        'config': config,
        'metrics': final_metrics,
        'threshold': best_threshold,
        'val_probs': val_probs,
        'val_labels': val_labels
    }

    

In [ ]:
print(results['Config_1']['metrics'])
print(results['Config_2']['metrics'])

y_proba_nn = results['Config_2']['val_probs']

In [ ]:
def optimize_and_compare_models(models_dict):
    """
    為每個模型找出最佳閾值並比較結果
    
    models_dict = {
        'model_name': {
            'y_proba': probabilities,
            'y_test': true_labels
        }
    }
    """
    comparison_results = []
    
    for model_name, data in models_dict.items():
        # 使用evaluate_threshold找出最佳閾值
        results_df = evaluate_threshold(data['y_test'], data['y_proba'])
        
        # 找出最佳成本節省的閾值
        best_idx = results_df['cost_savings'].idxmax()
        best_result = results_df.loc[best_idx]
        
        # 使用最佳閾值的預測結果
        y_pred_best = (data['y_proba'] >= best_result['threshold']).astype(int)
        tn, fp, fn, tp = confusion_matrix(data['y_test'], y_pred_best).ravel()
        
        accuracy = accuracy_score(data['y_test'], y_pred_best)
        roc_auc = roc_auc_score(data['y_test'], data['y_proba'])
        
        # 計算詳細成本
        no_show_cost = 200  # 每個未檢測到的no-show成本
        intervention_cost = 20  # 每次干預成本
        
        total_savings = (tp * no_show_cost) - ((fp + tp) * intervention_cost)
        missed_cost = fn * no_show_cost
        intervention_total_cost = (fp + tp) * intervention_cost
        
        comparison_results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Best Threshold': best_result['threshold'],
            'Recall (No-show)': tp / (tp + fn),
            'Precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'ROC-AUC': roc_auc,
            'True Positives': tp,
            'False Positives': fp,
            'False Negatives': fn,
            'Cost Savings ($)': total_savings,
            'Missed No-show Cost ($)': missed_cost,
            'Intervention Cost ($)': intervention_total_cost
        })
    
    # 創建比較表格
    comparison_df = pd.DataFrame(comparison_results)
    
    return comparison_df

# 準備模型結果
models_dict = {
    'Logistic Regression': {
        'y_proba': y_proba_lg_best,
        'y_test': y_test
    },
    'SVM': {
        'y_proba': y_proba_svm,
        'y_test': y_test
    },

    'Random Forest (Best)': {
        'y_proba': y_proba_rf_weighted_best,
        'y_test': y_test
    },
    
    'XGBoost (Best)': {
        'y_proba': y_proba_xgb_best,
        'y_test': y_test
    },
    'XGBoost (Top 10 Features)': {
        'y_proba': y_proba_top_xgb,
        'y_test': y_test
    },
    'XGBoost (Feature Enhanced)': {
        'y_proba': y_proba_best_enhanced,
        'y_test': y_test
    }, 
    'DNN': {
        'y_proba': y_proba_nn,
        'y_test': y_test
    }
}

# Generate comparison table    
comparison_df = optimize_and_compare_models(models_dict)

# Display results
print("\nModel Comparison with Optimized Thresholds:")
print(comparison_df.sort_values('Cost Savings ($)', ascending=False))

# Calculate total cost savings
total_savings = comparison_df['Cost Savings ($)'].max()
print(f"\nMaximum Potential Cost Savings: ${total_savings:,.2f}")


In [ ]:
# print(comparison_df)
comparison_table = comparison_df[['Model', 'Accuracy', 'Precision', 'Recall (No-show)', 'ROC-AUC', 'Cost Savings ($)']]
print(comparison_table)
def format_comparison_table(comparison_df):
    # 複製需要的列並重命名
    formatted_df = comparison_df[['Model', 'Accuracy', 'Precision', 'Recall (No-show)', 'ROC-AUC', 'Cost Savings ($)']].copy()
    formatted_df = formatted_df.rename(columns={'Recall (No-show)': 'Recall'})
    
    # 格式化數值
    formatted_df['Accuracy'] = formatted_df['Accuracy'].apply(lambda x: f"~{x:.0%}")
    formatted_df['Precision'] = formatted_df['Precision'].apply(lambda x: f"~{x:.2f}")
    formatted_df['Recall'] = formatted_df['Recall'].apply(lambda x: f"~{x:.2f}")
    formatted_df['ROC-AUC'] = formatted_df['ROC-AUC'].apply(lambda x: f"~{x:.2f}")
    formatted_df['Cost Savings ($)'] = formatted_df['Cost Savings ($)'].apply(lambda x: f"~{x:,.0f}")
    
    # 設置表格樣式
    styled_df = formatted_df.style.set_properties(**{
        'text-align': 'center',
        'padding': '8px'
    }).set_table_styles([{
        'selector': 'th',
        'props': [
            ('text-align', 'center'),
            ('font-weight', 'bold'),
            ('padding', '8px')
        ]
    }])
    
    return styled_df

# 2. 顯示格式化的表格
formatted_table = format_comparison_table(comparison_table)
display(formatted_table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

colors = plt.cm.tab10(np.linspace(0, 1, len(comparison_df)))

# Convert necessary columns to float if not already
comparison_df['Cost Savings ($)'] = comparison_df['Cost Savings ($)'].astype(float)
comparison_df['Best Threshold'] = comparison_df['Best Threshold'].astype(float)
comparison_df['Recall (No-show)'] = comparison_df['Recall (No-show)'].astype(float)
comparison_df['Precision'] = comparison_df['Precision'].astype(float)

# Plotting
fig, axs = plt.subplots(2, 2, figsize=(16, 10))

# 1. Cost Savings Comparison
axs[0, 0].barh(comparison_df['Model'], comparison_df['Cost Savings ($)'], color='#2E86C1')
axs[0, 0].set_title('Figure 4A. Cost Savings Comparison by Model', fontsize=15)
axs[0, 0].set_xlabel('Cost Savings ($)', fontsize=15)
axs[0, 0].invert_yaxis()  # Highest at top
axs[0, 0].grid(axis='x', linestyle='--', alpha=0.7)

# 2. Threshold vs Recall
axs[0, 1].scatter(comparison_df['Best Threshold'], comparison_df['Recall (No-show)'], color= colors,s=100)
axs[0, 1].set_xlim(0, 0.7)
axs[0, 1].set_ylim(0.60, 0.80)

for i, row in comparison_df.iterrows():
    axs[0, 1].annotate(row['Model'], (row['Best Threshold'], row['Recall (No-show)']), fontsize=15,)
axs[0, 1].set_title('Figure 4B. Threshold vs Recall', fontsize=15)
axs[0, 1].set_xlabel('Best Threshold', fontsize=15)
axs[0, 1].set_ylabel('Recall (No-show)', fontsize=15)
axs[0, 1].grid(True, linestyle='--', alpha=0.7)

# 3. Recall vs Precision (PR positioning)
axs[1, 0].scatter(comparison_df['Precision'], comparison_df['Recall (No-show)'], s=100, color=colors)
axs[1, 0].set_xlim(0.1, 0.32)
axs[1, 0].set_ylim(0.62, 0.80)
for i, row in comparison_df.iterrows():
    axs[1, 0].annotate(row['Model'], (row['Precision'], row['Recall (No-show)']), fontsize=15)
axs[1, 0].set_title('Precision vs Recall')
axs[1, 0].set_xlabel('Precision', fontsize=15)
axs[1, 0].set_ylabel('Recall', fontsize=15)
axs[1, 0].grid(True, linestyle='--', alpha=0.7)

# 4. Threshold Histogram
axs[1, 1].hist(comparison_df['Best Threshold'], bins=np.linspace(0.1, 0.5, 9), color='teal', edgecolor='black')
axs[1, 1].set_title('Distribution of Best Thresholds', fontsize=15)
axs[1, 1].set_xlabel('Best Threshold', fontsize=15)
axs[1, 1].set_ylabel('Number of Models', fontsize=15)
axs[1, 1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
